In [2]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr


c:\Users\Kirill\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_rez = pd.read_csv('df_rez_new.csv')[['id', 'resume_title', 'text_clean', 'skills_list', 'experience_text']]
df_vac = pd.read_csv('df_vac_new.csv')[['id', 'vacancy_name', 'text_clean', 'skills_final', 'experience_years_min']]

In [4]:
def restore_list(value):
    return ast.literal_eval(value)

df_rez['skills_list'] = df_rez['skills_list'].apply(restore_list)
df_vac['skills_final'] = df_vac['skills_final'].apply(restore_list)

## Псевдоэталон

In [5]:
def calc_exp_score(resume_exp, vacancy_exp):
    if pd.isna(resume_exp):
        resume_exp = 0.0
    if pd.isna(vacancy_exp):
        vacancy_exp = 0.0

    if vacancy_exp == 0:
        return 1.0

    if resume_exp >= vacancy_exp:
        return 1.0
    
    return resume_exp / vacancy_exp

In [6]:
def calc_skill_score(resume_skills, vacancy_skills):
    matched_skills = len(set(resume_skills) & set(vacancy_skills))
    skill_score = matched_skills / len(vacancy_skills)
    return skill_score, matched_skills

In [7]:
df_pairs = df_rez.merge(df_vac, how='cross')

In [ ]:
df_pairs[['skill_score', 'matched_skills']] = df_pairs.apply(
    lambda row: pd.Series(
        calc_skill_score(row['skills_list'], row['skills_final'])
    ),
    axis=1
)

In [ ]:
df_pairs['exp_score'] = df_pairs.apply(
    lambda row: calc_exp_score(row['experience_text'], row['experience_years_min']),
    axis=1
)

In [ ]:
df_pairs['pseudo_score'] = (
    0.67 * df_pairs['skill_score'] +
    0.33 * df_pairs['exp_score']
)

## Построение baseline

In [ ]:
train_resume_ids, test_resume_ids = train_test_split(
    df_rez['id'],
    test_size=0.2,
    random_state=42
)

train_pairs = df_pairs[df_pairs['id_x'].isin(train_resume_ids)].copy()
test_pairs = df_pairs[df_pairs['id_x'].isin(test_resume_ids)].copy()

train_pairs.shape, test_pairs.shape

((13783536, 14), (3445884, 14))

In [ ]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2
)

all_texts = pd.concat([
    df_rez['text_clean'],
    df_vac['text_clean']
])

tfidf_matrix = tfidf.fit_transform(all_texts)

resume_tfidf = tfidf_matrix[:len(df_rez)]
vacancy_tfidf = tfidf_matrix[len(df_rez):]

tfidf_sim_matrix = cosine_similarity(resume_tfidf, vacancy_tfidf)

In [ ]:
def get_top_matches(sim_matrix, df_rez_base, df_vac_base, top_k=5):
    results = []

    for i in range(sim_matrix.shape[0]):
        scores = sim_matrix[i]
        top_idx = np.argsort(scores)[::-1][:top_k]

        for rank, j in enumerate(top_idx, start=1):
            results.append({
                'resume_id': df_rez_base.iloc[i]['id'],
                'resume_title': df_rez_base.iloc[i]['resume_title'],
                'vacancy_id': df_vac_base.iloc[j]['id'],
                'vacancy_name': df_vac_base.iloc[j]['vacancy_name'],
                'rank': rank,
                'model_score': scores[j]
            })

    return pd.DataFrame(results)

In [ ]:
tfidf_top_matches = get_top_matches(
    tfidf_sim_matrix,
    df_rez,
    df_vac,
    top_k=5
)

tfidf_top_matches = tfidf_top_matches[
    tfidf_top_matches['resume_id'].isin(test_resume_ids)
].copy()

tfidf_top_matches.head()

,resume_id,resume_title,vacancy_id,vacancy_name,rank,model_score
0,2,Аналитик данных,15447,Аналитик данных,1,0.105119
1,2,Аналитик данных,6454,BI-аналитик,2,0.094277
2,2,Аналитик данных,1454,Старший аналитик (data analyst / data scientis...,3,0.089827
3,2,Аналитик данных,1427,Аналитик данных,4,0.089805
4,2,Аналитик данных,3679,Аналитик данных (CRM),5,0.085324


In [ ]:
tfidf_eval = tfidf_top_matches.merge(
    df_pairs[['id_x', 'id_y', 'skill_score', 'exp_score', 'pseudo_score']],
    left_on=['resume_id', 'vacancy_id'],
    right_on=['id_x', 'id_y'],
    how='left'
)

tfidf_eval.head()

,resume_id,resume_title,vacancy_id,vacancy_name,rank,model_score,id_x,id_y,skill_score,exp_score,pseudo_score
0,2,Аналитик данных,15447,Аналитик данных,1,0.105119,2,15447,0.105263,1.0,0.400526
1,2,Аналитик данных,6454,BI-аналитик,2,0.094277,2,6454,0.250000,1.0,0.497500
2,2,Аналитик данных,1454,Старший аналитик (data analyst / data scientis...,3,0.089827,2,1454,0.277778,1.0,0.516111
3,2,Аналитик данных,1427,Аналитик данных,4,0.089805,2,1427,0.000000,1.0,0.330000
4,2,Аналитик данных,3679,Аналитик данных (CRM),5,0.085324,2,3679,0.250000,1.0,0.497500


In [ ]:
mean_pseudo_score_5 = tfidf_eval['pseudo_score'].mean()
mean_skill_score_5 = tfidf_eval['skill_score'].mean()
mean_exp_score_5 = tfidf_eval['exp_score'].mean()

coverage_5 = (
    tfidf_eval['vacancy_id'].nunique() /
    df_vac['id'].nunique()
)

In [ ]:
tfidf_metrics = {
    'model': 'TF-IDF + cosine',
    'Mean Pseudo Score@5': mean_pseudo_score_5,
    'Mean Skill Score@5': mean_skill_score_5,
    'Mean Exp Score@5': mean_exp_score_5,
    'Coverage@5': coverage_5
}

tfidf_metrics

{'model': 'TF-IDF + cosine',
 'Mean Pseudo Score@5': np.float64(0.4358300120310526),
 'Mean Skill Score@5': np.float64(0.2337582744132063),
 'Mean Exp Score@5': np.float64(0.8460968732551646),
 'Coverage@5': 0.6285516285516286}

In [ ]:
test_resume_positions = [
    i for i, resume_id in enumerate(df_rez['id'])
    if resume_id in set(test_resume_ids)
]

test_tfidf_scores = tfidf_sim_matrix[test_resume_positions].ravel()

test_pseudo_scores = (
    df_pairs[df_pairs['id_x'].isin(test_resume_ids)]
    .sort_values(['id_x', 'id_y'])['pseudo_score']
    .values
)

tfidf_scores_long = []

for i in test_resume_positions:
    resume_id = df_rez.iloc[i]['id']
    
    temp = pd.DataFrame({
        'resume_id': resume_id,
        'vacancy_id': df_vac['id'].values,
        'model_score': tfidf_sim_matrix[i]
    })
    
    tfidf_scores_long.append(temp)

tfidf_scores_long = pd.concat(tfidf_scores_long, ignore_index=True)

tfidf_scores_eval = tfidf_scores_long.merge(
    df_pairs[['id_x', 'id_y', 'pseudo_score']],
    left_on=['resume_id', 'vacancy_id'],
    right_on=['id_x', 'id_y'],
    how='left'
)

spearman_corr = spearmanr(
    tfidf_scores_eval['model_score'],
    tfidf_scores_eval['pseudo_score']
)

spearman_corr

SignificanceResult(statistic=np.float64(0.3777336227183146), pvalue=np.float64(0.0))

### TF-IDF + Логистическая регрессия

In [ ]:
def make_logreg_data_by_percent(
    df_pairs,
    train_resume_ids,
    pos_pct=0.05,
    neg_pct=0.20
):
    train_source = df_pairs[df_pairs['id_x'].isin(train_resume_ids)].copy()

    train_source['rank_pct'] = (
        train_source
        .groupby('id_x')['pseudo_score']
        .rank(pct=True, ascending=False)
    )

    train_logreg = train_source[
        (train_source['rank_pct'] <= pos_pct) |
        (train_source['rank_pct'] >= 1 - neg_pct)
    ].copy()

    train_logreg['target'] = (
        train_logreg['rank_pct'] <= pos_pct
    ).astype(int)

    return train_logreg

In [ ]:
train_logreg = make_logreg_data_by_percent(
    df_pairs=df_pairs,
    train_resume_ids=train_resume_ids,
    pos_pct=0.05,
    neg_pct=0.20
)

train_logreg['target'].value_counts(normalize=True)

target
0    0.760451
1    0.239549
Name: proportion, dtype: float64

In [ ]:
print('Размер train_logreg:', train_logreg.shape)

train_logreg['target'].value_counts()

Размер train_logreg: (2927101, 16)


target
0    2225918
1     701183
Name: count, dtype: int64

In [ ]:
train_logreg[[
    'id_x',
    'id_y',
    'resume_title',
    'vacancy_name',
    'pseudo_score',
    'rank_pct',
    'target'
]].head(10)

,id_x,id_y,resume_title,vacancy_name,pseudo_score,rank_pct,target
1446,3,162,Аналитик данных,Data Engineer по построению DWH,0.408824,0.889120,0
1450,3,88,Аналитик данных,Data Scientist в команду рисков МСФО,1.000000,0.030146,1
1469,3,388,Аналитик данных,Data Engineer,0.385833,0.925502,0
1472,3,428,Аналитик данных,Data Scientist риск-модели,1.000000,0.030146,1
1475,3,438,Аналитик данных,Product/data analyst,1.000000,0.030146,1
1476,3,457,Аналитик данных,Data Scientist,0.404444,0.897436,0
1479,3,547,Аналитик данных,Аналитик данных,1.000000,0.030146,1
1480,3,2641,Аналитик данных,Data Engineer,0.441667,0.819127,0
1481,3,568,Аналитик данных,Аналитик данных по работе с маркетплейсами (ст...,0.330000,0.969508,0
1482,3,572,Аналитик данных,Bi-аналитик PowerBI,0.330000,0.969508,0


In [ ]:
train_logreg['pair_text'] = (
    train_logreg['text_clean_x'].fillna('') +
    ' [SEP] ' +
    train_logreg['text_clean_y'].fillna('')
)

train_logreg[['id_x', 'id_y', 'target', 'pair_text']].head()

,id_x,id_y,target,pair_text
1446,3,162,0,апрель 2025 по termoland услуги для населения....
1450,3,88,1,апрель 2025 по termoland услуги для населения....
1469,3,388,0,апрель 2025 по termoland услуги для населения....
1472,3,428,1,апрель 2025 по termoland услуги для населения....
1475,3,438,1,апрель 2025 по termoland услуги для населения....


In [ ]:
sample_train_resume_ids = pd.Series(train_resume_ids).sample(
    n=2000,
    random_state=42
)

train_logreg = make_logreg_data_by_percent(
    df_pairs=df_pairs,
    train_resume_ids=sample_train_resume_ids,
    pos_pct=0.03,
    neg_pct=0.10
)

train_logreg.shape

(256817, 16)

In [ ]:
train_logreg['target'].value_counts(normalize=True)

target
0    0.651826
1    0.348174
Name: proportion, dtype: float64

In [ ]:
train_logreg['pair_text'] = (
    train_logreg['text_clean_x'].fillna('') +
    ' [SEP] ' +
    train_logreg['text_clean_y'].fillna('')
)

train_logreg[['id_x', 'id_y', 'target', 'pair_text']].head()

,id_x,id_y,target,pair_text
13058,14,906,1,июль 2024 по systeme digital москва systemedig...
13067,14,998,1,июль 2024 по systeme digital москва systemedig...
13146,14,2232,1,июль 2024 по systeme digital москва systemedig...
13171,14,2479,1,июль 2024 по systeme digital москва systemedig...
13173,14,2488,1,июль 2024 по systeme digital москва systemedig...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_logreg = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.9
)

X_train = tfidf_logreg.fit_transform(train_logreg['pair_text'])
y_train = train_logreg['target']

X_train.shape, y_train.shape

((256817, 20000), (256817,))

In [ ]:
logreg = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    solver='liblinear'
)

logreg.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [ ]:
test_pairs = df_pairs[df_pairs['id_x'].isin(test_resume_ids)].copy()

test_pairs['pair_text'] = (
    test_pairs['text_clean_x'].fillna('') +
    ' [SEP] ' +
    test_pairs['text_clean_y'].fillna('')
)


test_pairs.shape

NameError: name 'df_pairs' is not defined

## Архивная часть

In [218]:
df_rez.head()

,id,resume_title,text_clean,skills_list,experience_text
0,2,Аналитик данных,сентябрь 2024 сентябрь 2025 кофемания москва г...,"[пользователь пк, ms sql, ms office, driving l...",8.92
1,3,Аналитик данных,апрель 2025 по termoland услуги для населения....,"[ms powerpoint, python, numpy, sql, git, pycha...",6.50
2,4,Аналитик данных,декабрь 2024 по центральный банк российской фе...,"[data science, регрессионный анализ, проверка ...",2.67
3,6,Аналитик данных,январь 2025 по нева дельта спб аналитик сбор о...,"[sql, power bi, ms power bi, dax, tableau, abc...",8.17
4,8,Аналитик данных,октябрь 2011 по сбер москва rabota.sber.ru фин...,"[python, vba, sql, ms office, oracle, qlik sen...",14.42


In [220]:
df_vac.head()

,id,vacancy_name,text_clean,skills_final,experience_years_min
0,15,Аналитик данных / экономист инвестиционных про...,46 органов исполнительной власти 132 территори...,"[python, django, pandas, ms office, power bi, ...",1.0
1,34,Data Engineer (разработчик DWH),задаем тренды в технологиях ритейла x5 group —...,"[sql, python, big data, apache airflow, apache...",3.0
2,92,Аналитик данных,центр компетенций бизнес аналитики и финансово...,"[sql, python, hadoop, excel]",1.0
3,162,Data Engineer по построению DWH,компания ecofinance развивает и внедряет проду...,"[etl, sql, dwh, apache kafka, debezium, bi, db...",3.0
4,65,Senior Data Scientist,создавай инновации в атмосфере свободы — и рас...,"[python, ml-библиотеки, sql]",1.0


In [221]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4950.40it/s]


In [222]:
resume_texts = df_rez['text_clean'].tolist()
vacancy_texts = df_vac['text_clean'].tolist()

In [223]:
resume_embeddings = model.encode(
    resume_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

vacancy_embeddings = model.encode(
    vacancy_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 46/46 [00:28<00:00,  1.62it/s]


In [224]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(resume_embeddings, vacancy_embeddings)
sim_matrix.shape

(11940, 1443)

In [225]:
def get_top_matches(sim_matrix, df_rez_base, df_vac_base, top_k=5):
    results = []

    for i in range(sim_matrix.shape[0]):
        scores = sim_matrix[i]
        top_idx = np.argsort(scores)[::-1][:top_k]

        for rank, j in enumerate(top_idx, start=1):
            results.append({
                'resume_id': df_rez_base.iloc[i]['id'],
                'resume_title': df_rez_base.iloc[i]['resume_title'],
                'vacancy_id': df_vac_base.iloc[j]['id'],
                'vacancy_name': df_vac_base.iloc[j]['vacancy_name'],
                'rank': rank,
                'semantic_score': scores[j]
            })

    return pd.DataFrame(results)

In [226]:
top_matches = get_top_matches(sim_matrix, df_rez, df_vac, top_k=5)
top_matches.head(10)

,resume_id,resume_title,vacancy_id,vacancy_name,rank,semantic_score
0,2,Аналитик данных,10557,BI-аналитик,1,0.653296
1,2,Аналитик данных,12318,Data Analyst,2,0.628879
2,2,Аналитик данных,13671,Аналитик данных - статистик,3,0.628866
3,2,Аналитик данных,16748,Аналитик данных/статистик,4,0.626059
4,2,Аналитик данных,12712,Аналитик данных (эксперт Excel),5,0.609621
5,3,Аналитик данных,15243,Senior Data Scientist (ЦУНДО),1,0.688142
6,3,Аналитик данных,12197,Senior Data Scientist (ЦУНДО),2,0.688142
7,3,Аналитик данных,15015,Data Analyst (Middle),3,0.676931
8,3,Аналитик данных,11006,Senior Data Engineer,4,0.670128
9,3,Аналитик данных,15718,Аналитик данных/ Аналитик DWH,5,0.662332


## Оценка качества

In [227]:
def calc_skill_score(resume_skills, vacancy_skills):
    if not isinstance(resume_skills, list):
        resume_skills = []

    if not isinstance(vacancy_skills, list):
        vacancy_skills = []

    resume_skills = set([str(x).lower().strip() for x in resume_skills])
    vacancy_skills = set([str(x).lower().strip() for x in vacancy_skills])

    if len(vacancy_skills) == 0:
        return 0.0, 0

    matched_skills = len(resume_skills & vacancy_skills)
    skill_score = matched_skills / len(vacancy_skills)

    return skill_score, matched_skills


def calc_exp_score(resume_exp, vacancy_exp):
    if pd.isna(resume_exp):
        resume_exp = 0

    if pd.isna(vacancy_exp):
        vacancy_exp = 0

    resume_exp = float(resume_exp)
    vacancy_exp = float(vacancy_exp)

    if vacancy_exp == 0:
        return 1.0

    if resume_exp >= vacancy_exp:
        return 1.0

    return resume_exp / vacancy_exp

In [228]:
top_matches_eval = (
    top_matches
    .merge(
        df_rez[["id", "resume_title", "skills_list", "experience_text"]],
        left_on="resume_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"])
    .merge(
        df_vac[["id", "vacancy_name", "skills_final", "experience_years_min"]],
        left_on="vacancy_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"])
)

top_matches_eval.head()

,resume_id,resume_title_x,vacancy_id,vacancy_name_x,rank,semantic_score,resume_title_y,skills_list,experience_text,vacancy_name_y,skills_final,experience_years_min
0,2,Аналитик данных,10557,BI-аналитик,1,0.653296,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,BI-аналитик,"[sql, datalens, clickhouse, sucd, python, etl]",1.0
1,2,Аналитик данных,12318,Data Analyst,2,0.628879,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Data Analyst,"[английский — b1 — средний, python, sql, опыт ...",1.0
2,2,Аналитик данных,13671,Аналитик данных - статистик,3,0.628866,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Аналитик данных - статистик,"[анализ данных, статистика, сэд, ms office, ms...",0.0
3,2,Аналитик данных,16748,Аналитик данных/статистик,4,0.626059,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Аналитик данных/статистик,"[работа с большим объемом информации, аналитич...",0.0
4,2,Аналитик данных,12712,Аналитик данных (эксперт Excel),5,0.609621,Аналитик данных,"[пользователь пк, ms sql, ms office, driving l...",8.92,Аналитик данных (эксперт Excel),"[ms excel, vba/macros]",1.0


In [229]:
top_matches_eval[["skill_score", "matched_skills"]] = top_matches_eval.apply(
    lambda row: pd.Series(
        calc_skill_score(
            row["skills_list"],
            row["skills_final"]
        )
    ),
    axis=1
)

top_matches_eval["exp_score"] = top_matches_eval.apply(
    lambda row: calc_exp_score(
        row["experience_text"],
        row["experience_years_min"]
    ),
    axis=1
)

In [230]:
top_matches_eval[["skill_score", "matched_skills"]] = top_matches_eval.apply(
    lambda row: pd.Series(
        calc_skill_score(row["skills_list"], row["skills_final"])
    ),
    axis=1
)

top_matches_eval["exp_score"] = top_matches_eval.apply(
    lambda row: calc_exp_score(
        row["experience_text"],
        row["experience_years_min"]
    ),
    axis=1
)

In [231]:
top_matches_eval[[
    "resume_id",
    "vacancy_id",
    "rank",
    "semantic_score",
    "skill_score",
    "matched_skills",
    "exp_score"
]].head()

,resume_id,vacancy_id,rank,semantic_score,skill_score,matched_skills,exp_score
0,2,10557,1,0.653296,0.166667,1.0,1.0
1,2,12318,2,0.628879,0.200000,1.0,1.0
2,2,13671,3,0.628866,0.142857,2.0,1.0
3,2,16748,4,0.626059,0.133333,2.0,1.0
4,2,12712,5,0.609621,0.000000,0.0,1.0


In [232]:
# Среднее совпадение навыков в top-5
mean_skill_score_5 = (
    top_matches_eval[top_matches_eval["rank"] <= 5]["skill_score"]
    .mean()
)


# Good Match Rate@5
def good_match_rate_at_k(df, k=5, skill_threshold=0.3, exp_threshold=0.7):
    temp = df[df["rank"] <= k].copy()

    temp["is_good"] = (
        (temp["skill_score"] >= skill_threshold) &
        (temp["exp_score"] >= exp_threshold)
    )

    return temp.groupby("resume_id")["is_good"].max().mean()


good_match_rate_5 = good_match_rate_at_k(
    top_matches_eval,
    k=5,
    skill_threshold=0.3,
    exp_threshold=0.7
)


# Experience Fit Rate@5
def experience_fit_rate_at_k(df, k=5, exp_threshold=0.7):
    temp = df[df["rank"] <= k].copy()

    temp["exp_fit"] = temp["exp_score"] >= exp_threshold

    return temp.groupby("resume_id")["exp_fit"].mean().mean()


experience_fit_rate_5 = experience_fit_rate_at_k(
    top_matches_eval,
    k=5,
    exp_threshold=0.7
)


# Итоговая таблица
metrics_summary = pd.DataFrame({
    "metric": [
        "Good Match Rate@5",
        "Experience Fit Rate@5",
        "Mean skill_score@5"
    ],
    "value": [
        good_match_rate_5,
        experience_fit_rate_5,
        mean_skill_score_5
    ]
})

metrics_summary

,metric,value
0,Good Match Rate@5,0.593802
1,Experience Fit Rate@5,0.835511
2,Mean skill_score@5,0.227859


Mean Semantic Score@K

In [233]:
top_matches["semantic_score"].mean()

np.float32(0.6402104)

Score Gap@1-5

In [235]:
score_pivot = top_matches.pivot(
    index="resume_id",
    columns="rank",
    values="semantic_score"
)

(score_pivot[1] - score_pivot[5]).mean()

np.float32(0.042066287)

Strong Match Share@K

In [236]:
threshold = 0.5

(
    top_matches
    .groupby("resume_id")["semantic_score"]
    .max()
    .ge(threshold)
    .mean()
)

np.float64(0.9857621440536013)

Coverage@5

In [237]:
top_matches["vacancy_id"].nunique() / df_vac["id"].nunique()

0.5758835758835759

Vacancy Popularity Bias

In [238]:
top_matches["vacancy_id"].value_counts().head(10)

vacancy_id
15197    1474
14507    1312
13604    1071
12197    1006
15243     976
15041     967
3752      966
4363      765
13926     763
1900      625
Name: count, dtype: int64

Rank Stability / разница score внутри top-5

In [240]:
top_matches.groupby("resume_id")["semantic_score"].std().mean()

np.float32(0.017663924)